In [1]:
import shutil
from pathlib import Path
from typing import Literal

import instructor
import lancedb
import pandas as pd
from google.generativeai import GenerativeModel
from IPython.display import Markdown
from lancedb.rerankers import ColbertReranker

from dreamai.ai import ModelName
from dreamai.dialog import Dialog
from dreamai.dialog_models import StepBackQuestions, TableDescription
from dreamai.rag import add_to_lance_table, pdf_to_md_docs

%load_ext autoreload
%autoreload 2
%reload_ext autoreload

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [2]:
ask_kid = instructor.from_gemini(
    client=GenerativeModel(model_name=ModelName.GEMINI_FLASH)
)

In [3]:
shutil.rmtree("lance")

In [4]:
LANCE_URI = "lance/kid/"

In [5]:
lance_db = lancedb.connect(uri=LANCE_URI)
reranker = ColbertReranker("answerdotai/answerai-colbert-small-v1")

In [6]:
files = ["soccer.txt", "finance.txt"]
md_data = {Path(file).stem: pdf_to_md_docs(file_path=file) for file in files}

In [7]:
table_description_dialog = Dialog.load("dreamai/dialogs/table_description_dialog.json")
table_descriptions = []
for file in files:
    table_name = Path(file).stem
    table_descriptions.append(
        TableDescription(
            name=table_name,
            description=ask_kid.create(
                response_model=str,
                **table_description_dialog.gemini_kwargs(
                    template_data={
                        "database_name": table_name,
                        "sample_text": md_data[table_name].markdown,
                    }
                ),  # type: ignore
            ),
        )
    )
    _ = add_to_lance_table(
        db=lance_db, table_name=table_name, data=md_data[table_name].chunks
    )

/home/hamza/dev/dreamai/.venv/lib/python3.11/site-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [8]:
query = "what should I do to save money?"

In [9]:
table_selection_dialog = Dialog.load("dreamai/dialogs/table_selection_dialog.json")
table_names = Literal[
    *[table_description.name for table_description in table_descriptions]  # type: ignore
]
table_names

typing.Literal['soccer', 'finance']

In [10]:
table_name = ask_kid.create(
    response_model=table_names,
    **table_selection_dialog.gemini_kwargs(
        template_data={
            "query": query,
            "database_list": [
                table_description.model_dump_json(indent=2)
                for table_description in table_descriptions
            ],
        }
    ),  # type: ignore
)

In [11]:
table = lance_db.open_table(str(table_name))
table_name

'finance'

In [12]:
step_back_dialog = Dialog.load("dreamai/dialogs/step_back_dialog.json")

In [13]:
step_back_questions = ask_kid.create(
    response_model=StepBackQuestions,
    **step_back_dialog.gemini_kwargs(user=query),  # type: ignore
)

In [14]:
res = (
    pd.concat(
        [
            table.search(
                query=question,  # type: ignore
                query_type="hybrid",
            )
            .rerank(reranker=reranker)  # type: ignore
            .limit(3)
            .to_pandas()
            for question in step_back_questions.questions + [query]  # type: ignore
        ]
    )
    .drop_duplicates("text")
    .sort_values("_relevance_score", ascending=False)
    .reset_index(drop=True)
)

In [15]:
res

,vector,text,_relevance_score
0,"[0.016893987, -0.019692775, 0.010257134, 0.057...",## 11.2 Pay Yourself First\nMake saving and in...,0.906181
1,"[-0.0014483078, -0.013668669, 0.009310877, 0.0...",## 3.2 Tracking Expenses\nTracking your expens...,0.893962
2,"[0.022010049, 0.0017746452, 0.0023124204, 0.05...",**Categorize your expenses**: Group similar ex...,0.890944
3,"[0.019293692, -0.006808497, -0.016382698, 0.04...",## 11.3 Live Below Your Means\nSpending less t...,0.885745
4,"[0.008993197, -0.015739046, -0.0001547457, 0.0...",## 3.1 Creating a Budget\nA budget is a plan t...,0.856309
5,"[0.0065500527, -0.027039519, 0.0033820502, 0.0...",# 5. Risk Management and Insurance\nEffective ...,0.842812
6,"[0.0021942595, 0.01197012, 0.014916414, 0.0350...",## 6.2 Tax-advantaged Accounts\nUtilize tax-ad...,0.842144


In [16]:
Markdown("\n".join(res["text"]))

## 11.2 Pay Yourself First
Make saving and investing a priority by treating them as non-negotiable expenses.
- **Automate savings**: Set up automatic transfers to savings and investment accounts.
- **Increase savings with income**: When you get a raise, increase your savings rate
before lifestyle inflation sets in.
- **Use tax-advantaged accounts**: Maximize contributions to 401(k)s, IRAs, and HSAs
where possible.
## 3.2 Tracking Expenses
Tracking your expenses is crucial for sticking to your budget and understanding your
spending habits. Consider these methods:
- **Use a spreadsheet**: Create a simple spreadsheet to manually record your expenses.
- **Use budgeting apps**: Apps like Mint, YNAB, or Personal Capital can automatically
categorize your expenses.
- **Envelope system**: Use cash envelopes for different expense categories to limit
overspending.
Regularly review your expenses to identify areas where you might be overspending and
opportunities to save.
**Categorize your expenses**: Group similar expenses together (e.g., housing,
transportation, food).
4. **Set spending limits for each category**: Based on your income and financial
goals, allocate a specific amount to each category.
5. **Review and adjust**: Regularly review your budget and make adjustments as needed.
Remember, a budget is not meant to restrict you, but to empower you to make conscious
decisions about your money.
## 11.3 Live Below Your Means
Spending less than you earn is fundamental to financial success.
- **Create and stick to a budget**: Track your spending and ensure it aligns with your
financial goals.
- **Differentiate between needs and wants**: Prioritize essential expenses and be
mindful of discretionary spending.
- **Avoid lifestyle inflation**: Resist the urge to increase spending as your income
grows.
## 3.1 Creating a Budget
A budget is a plan that helps you balance your income and expenses. Here's how to
create an effective budget:
1. **Calculate your total income**: Include all sources of income, such as salary,
investments, and any side hustles.
2. **List all your expenses**: Start with fixed expenses (rent, utilities, loan
payments) and then add variable expenses (groceries, entertainment, etc.).
3. **Categorize your expenses**: Group similar expenses together (e.g., housing,
transportation, food).
4. **Set spending limits for each category**: Based on your income and financial
goals, allocate a specific amount to each category.
5. **Review and adjust
# 5. Risk Management and Insurance
Effective risk management is a crucial part of financial planning. Insurance plays a
key role in protecting your assets and income from unforeseen events. Understanding
different types of insurance and assessing your needs can help you create a
comprehensive risk management strategy.
## 5.1 Types of Insurance
There are several types of insurance that can be part of your financial plan:
- Health Insurance
- Life Insurance
- Disability Insurance
- Property Insurance (Homeowners/Renters)
- Auto Insurance
- Liability Insurance
- Long-term Care Insurance
Each type serves a specific purpose in protecting you and your assets from different
risks.
## 6.2 Tax-advantaged Accounts
Utilize tax-advantaged accounts to reduce your current or future tax liability:
- **Traditional 401(k) and IRA**: Contributions are made with pre-tax dollars,
reducing your current taxable income. Taxes are paid upon withdrawal.
- **Roth 401(k) and IRA**: Contributions are made with after-tax dollars, but
qualified withdrawals are tax-free.
- **Health Savings Account (HSA)**: Triple tax advantage - contributions are taxdeductible, grow tax-free, and withdrawals for qualified medical expenses are taxfree.
- **529 College Savings Plans**: Contributions grow tax-free and withdrawals for
qualified education expenses are tax-free.